In [1]:
import pandas as pd
from src.structural_functions.goals_calculation import gen_goals_calculations,  gen_dataviews
#from agents_langchain import gen_goals

In [2]:
ongoing_df = pd.read_parquet(f'files\mkt_ongoing.parquet')


## Calculo de metas
##### retorna dicionário de valores das metas calculas por indicadores
#### classe de indicadores: ['NPS', 'Impressão Promotora', 'Impressão Detratora', 'Protagonismo', 'Frequência', 'Valoração', 'Contr. ao NPS']
#### indicadores: 
##### | NPS -> ['NPS - Média mensal', 'NPS - Acumulado no ano']
##### | Impressão Promotora -> ['Impressão Promotora - Média mensal', 'Impressão Promotora - Acumulado no ano', 'Impressão Promotora (%) - Média mensal']
##### | Impressão Detratora -> ['Impressão Detratora - Média mensal', 'Impressão Detratora - Acumulado no ano', 'Impressão Detratora (%) - Média mensal']
##### | Protagonismo -> ['Protagonismo - Média mensal', 'Protagonismo - Acumulado no ano', 'Protagonismo Promotor - Média mensal']
##### | Frequência -> ['Cadência de Publicações - Média por dia', 'Cadência de Publicações Promotora - Média por dia', 'Publicações - Acumulado no ano', 'Publicações Promotora - Acumulado no ano']
##### | Valoração -> ['Valoração - Média mensal', 'Valoração - Acumulado no ano']
##### | Contr. ao NPS -> ações de comunicação -> ['(%) Contr. das Ações ao NPS - Média mensal', '(%) Contr. das Ações ao NPS - Acumulado no ano', '(%) Contr. das Ações Promotoras ao NPS - Média mensal', '(%) Contr. das Ações Promotoras ao NPS - Acumulado no ano']

In [8]:
dict_goals_calculations = gen_goals_calculations('files\mkt_ongoing.parquet')
dict_goals_calculations

{'NPS': {'NPS - Média mensal': {'descricao': 'O indicador é a média mensal do NPS. É ideal para observar o desempenho mensal, evitando desvios importantes no NPS e riscos de percepção da imagem da marca no período.',
   'baseline': 87.33,
   'conservadora': {'meta': 83.23, 'variacao': -4.1},
   'moderada': {'meta': 89.08, 'variacao': 1.75},
   'ousada': {'meta': 93.76, 'variacao': 6.43}},
  'NPS - Acumulado no ano': {'descricao': 'O indicador é o agregado do NPS no decorrer do ano. É ideal para observar o total da percepção no acumulada do ano sobre a marca.',
   'baseline': 78.0,
   'conservadora': {'meta': 76.0, 'variacao': -2.0},
   'moderada': {'meta': 80.0, 'variacao': 2.0},
   'ousada': {'meta': 82.0, 'variacao': 4.0}}},
 'Impressão Promotora': {'Impressão Promotora - Média mensal': {'descricao': 'O indicador é a média mensal do impacto promotor da marca. É ideal para observar o desempenho mensal da percepção promotora, com o objetivo de reduzir desvios importantes e manter a pro

## Função para gerar as metas 
### retorna um dicionário com 



In [3]:
dict_goals = gen_goals('mkt')

In [4]:
dict_goals

{'objectives': [{'objectives': 'Reforçar a percepção de liderança da Cortex Intelligence no uso de Inteligência Artificial para Go-to-Market na América Latina.',
   'index': ['Protagonismo - Média mensal',
    'Protagonismo Promotor - Média mensal',
    'Impressão Promotora - Média mensal'],
   'audience impact': ['Despertar consciência para marca',
    'Suscitar interesse',
    'Garantir Identificação e empatia']},
  {'objectives': 'Mitigar riscos de associação indevida ao uso invasivo de dados e reforçar o compromisso ético da Cortex Intelligence.',
   'index': ['Impressão Detratora - Média mensal',
    '(%) Contr. das Ações ao NPS - Média mensal',
    'Cadência de Publicações Promotora - Média por dia'],
   'audience impact': ['Proporcionar Conhecimento',
    'Garantir Identificação e empatia',
    'Estabelecer Interação']},
  {'objectives': "Destacar as inovações e soluções diferenciadas da Cortex Intelligence, reforçando seu valor 'People First'.",
   'index': ['Publicações Promot

### onboading

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, datetime
from typing import Any, Optional, Union

Number = Union[int, float]


def _to_date(d: Union[str, date, datetime]) -> date:
    """Aceita date/datetime/ISO string (YYYY-MM-DD)."""
    if isinstance(d, date) and not isinstance(d, datetime):
        return d
    if isinstance(d, datetime):
        return d.date()
    if isinstance(d, str):
        # tenta ISO primeiro (YYYY-MM-DD)
        try:
            return datetime.fromisoformat(d[:10]).date()
        except ValueError as e:
            raise ValueError(f"Data inválida: {d!r}. Use 'YYYY-MM-DD' ou date/datetime.") from e
    raise TypeError(f"Tipo inválido para data: {type(d).__name__}")


def valoracao_calc(
    *,
    data: Union[str, date, datetime],
    midia: str,
    alcance_organico: Optional[Number],
    tipo_midia_post: Optional[str] = None,     # ex: "photo", "album", "link", "video", "status", "text"
    duracao_video: Optional[Number] = None,    # segundos
    valoracao_radio_tv: Optional[Number] = None,
) -> float:
    """
    Implementa a lógica da "versão 2025.01.06" exatamente como no script.
    Retorna 0.0 quando não houver regra aplicável.

    Regras por data:
      - data < 2020-01-01: regras antigas (Facebook por faixas, Instagram por faixas, etc.)
      - data < 2024-12-01: regras intermediárias (Facebook por tipo, Instagram power, etc.)
      - data > 2024-11-30: regras mais recentes (inclui album/text e TV)
    """
    d = _to_date(data)
    midia = (midia or "").strip()
    tipo = (tipo_midia_post or "").strip().lower()

    # se alcance for None/NaN, trata como 0 (mantém retorno 0 no final)
    if alcance_organico is None:
        alcance = None
    else:
        try:
            alcance = float(alcance_organico)
        except (TypeError, ValueError):
            alcance = None

    def per_mil(mult: float) -> float:
        if alcance is None:
            return 0.0
        return (alcance / 1000.0) * float(mult)

    # ----------------- BLOCO 1: antes de 2020-01-01 -----------------
    if d < date(2020, 1, 1):
        if midia == "Facebook":
            if alcance is None:
                return 0.0
            if alcance < 2700:
                return (alcance / 1000.0) * 2.70
            elif 2700 <= alcance < 7200:
                return (alcance / 1000.0) * 2.93
            elif 7200 <= alcance < 19000:
                return (alcance / 1000.0) * 3.21
            elif 19000 <= alcance < 51000:
                return (alcance / 1000.0) * 4.06
            elif 51000 <= alcance < 130000:
                return (alcance / 1000.0) * 5.52
            elif 130000 <= alcance < 340000:
                return (alcance / 1000.0) * 7.66
            elif alcance >= 340000:
                return (alcance / 1000.0) * 9.92

        elif midia == "Instagram":
            if alcance is None:
                return 0.0
            if alcance < 2100:
                return (alcance / 1000.0) * 0.74
            elif 2100 <= alcance < 5500:
                return (alcance / 1000.0) * 0.79
            elif 5500 <= alcance < 14000:
                return (alcance / 1000.0) * 0.72
            elif 14000 <= alcance < 37000:
                return (alcance / 1000.0) * 0.90
            elif 37000 <= alcance < 99000:
                return (alcance / 1000.0) * 0.93
            elif 99000 <= alcance < 263000:
                return (alcance / 1000.0) * 1.14
            elif alcance >= 263000:
                return (alcance / 1000.0) * 5.05

        elif midia == "YouTube":
            return per_mil(56)

        elif midia == "Twitter":
            return per_mil(19.6)

        elif midia == "Online":
            return per_mil(33.27)

        elif midia == "Impresso":
            return per_mil(106.25)

        return 0.0

    # ----------------- BLOCO 2: antes de 2024-12-01 -----------------
    if d < date(2024, 12, 1):
        if midia == "Facebook" and tipo == "photo":
            return per_mil(13.38)

        elif midia == "Facebook" and tipo == "link":
            return per_mil(13.40)

        elif midia == "Facebook" and tipo == "video":
            return per_mil(11.22)

        elif midia == "Facebook" and tipo == "status":
            return per_mil(12.87)

        elif midia == "Instagram":
            # ([Alcance]/1000)*(322*(Alcance^-0.268))
            if alcance is None or alcance <= 0:
                return 0.0
            return (alcance / 1000.0) * (322.0 * (alcance ** (-0.268)))

        elif midia == "YouTube":
            if alcance is None:
                return 0.0
            dur = float(duracao_video or 0)
            if dur <= 269:
                return per_mil(75)
            elif dur <= 299:
                return per_mil(80)
            elif dur <= 359:
                return per_mil(85)
            elif dur <= 419:
                return per_mil(90)
            elif dur <= 509:
                return per_mil(95)
            else:
                return per_mil(100)

        elif midia == "Twitter":
            return per_mil(24.61)

        elif midia == "Online":
            if alcance is None:
                return 0.0
            return min((alcance / 1000.0) * 190.03, 146818.0)

        elif midia == "Impresso":
            if alcance is None or alcance <= 0:
                return 0.0
            val = 530225.0 * (alcance ** (-0.584))
            if val > 3229.12:
                return per_mil(3229.12)
            elif val < 90.34:
                return per_mil(90.34)
            else:
                return (alcance / 1000.0) * val

        return 0.0

    # ----------------- BLOCO 3: depois de 2024-11-30 -----------------
    # (equivale a d >= 2024-12-01)
    if midia == "Facebook" and (tipo == "photo" or tipo == "album"):
        return per_mil(13.38)

    elif midia == "Facebook" and tipo == "link":
        return per_mil(13.40)

    elif midia == "Facebook" and tipo == "video":
        return per_mil(11.22)

    elif midia == "Facebook" and (tipo == "status" or tipo == "text"):
        return per_mil(12.87)

    elif midia == "Instagram":
        if alcance is None or alcance <= 0:
            return 0.0
        return (alcance / 1000.0) * (322.0 * (alcance ** (-0.268)))

    elif midia == "YouTube":
        if alcance is None:
            return 0.0
        dur = float(duracao_video or 0)
        if dur <= 269:
            return per_mil(75)
        elif dur <= 299:
            return per_mil(80)
        elif dur <= 359:
            return per_mil(85)
        elif dur <= 419:
            return per_mil(90)
        elif dur <= 509:
            return per_mil(95)
        else:
            return per_mil(100)

    elif midia == "Twitter":
        return per_mil(24.61)

    elif midia == "Online":
        if alcance is None:
            return 0.0
        return min((alcance / 1000.0) * 190.03, 146818.0)

    elif midia == "Impresso":
        if alcance is None or alcance <= 0:
            return 0.0
        val = 530225.0 * (alcance ** (-0.584))
        if val > 3229.12:
            return per_mil(3229.12)
        elif val < 90.34:
            return per_mil(90.34)
        else:
            return (alcance / 1000.0) * val

    elif midia == "TV":
        try:
            return float(valoracao_radio_tv or 0.0)
        except (TypeError, ValueError):
            return 0.0

    return 0.0

In [3]:
import pandas as pd

In [24]:

def transform_dataframe(path_onboarding_dataframe: str) -> pd.DataFrame: 

    dataframe = pd.read_parquet(f'df_classificado.parquet')
    dataframe = dataframe.rename(columns={'Protagonismo' : 'Nível de Protagonismo final', 
                                        'tags_positividade' : 'Sentimento',
                                        'tags_promocao' : 'Tipos de impactos',
                                        'Visualizações' : 'Alcance orgânico',
                                        'marcas' : 'Empresa analisada'})

    dataframe['Mídia'] = 'Online'
    dataframe['Veículo'] = dataframe['Veículo (Default)']  
    dataframe['Produto analisado'] = 'None'
    dataframe['Tier'] = 'Tier 1'
    dataframe['Jornalista'] = 'Outros'
    dataframe['Ação'] =  'Outros'
    dataframe['Tipo da ação'] =  'Outros'
    dataframe['Status classificação'] = 'Classificado'
    dataframe['Valoração'] = dataframe.apply(
        lambda r: valoracao_calc(
            data=r['Data'],
            midia=r['Mídia'],
            alcance_organico=r['Alcance orgânico']    ),
        axis=1
    )

    prot_map = {'Coadjuvante' : 'Citação relevante', 
                'Muito Protagonista' : 'Protagonismo', 
                'Protagonista' : 'Protagonismo', 
                'Figurante' : 'Figurante'}

    dataframe['Nível de Protagonismo final'] = dataframe['Nível de Protagonismo final'].apply(lambda x : prot_map[x])

    map_impacto = {'Promotor' : 'Promotores', 
                'Inócuo' : 'Inócuos',
                'Detrator' : 'Detratores'

    }

    dataframe['Tipos de impactos'] = dataframe['Tipos de impactos'].apply(lambda x : map_impacto[x])
    path_name = path_onboarding_dataframe.split('.parquet')[0]
    
    dataframe.to_parquet(path_name + '_transformed' + '.parquet', index=False)
    return dataframe

In [29]:
transform_dataframe(r'./files/df_classificado.parquet')

,Post_ID,Link,Título,Conteúdo,Veículo (Default),Alcance orgânico,Data,agregado,Empresa analisada,frases,...,Tipos de impactos,Mídia,Veículo,Produto analisado,Tier,Jornalista,Ação,Tipo da ação,Status classificação,Valoração
0,1.373964e+19,https://www.uol.com.br/carros/colunas/paula-ga...,Imposto Seletivo: item da reforma tributária p...,A reforma tributária colocou a indústria auto...,Uol,4828868.0,2026-02-10,Imposto Seletivo: item da reforma tributária p...,byd,"Um exemplo citado é o de Alexandre Baldy, vice...",...,Inócuos,Online,Uol,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
1,2.975351e+18,https://www.terra.com.br/noticias/byd-projeta-...,BYD projeta produção inicial de baterias sólid...,A BYD intensificou seus investimentos em tecn...,Terra Brasil,2595578.0,2026-02-10,BYD projeta produção inicial de baterias sólid...,byd,BYD projeta produção inicial de baterias sólid...,...,Promotores,Online,Terra Brasil,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
2,1.396257e+19,https://g1.globo.com/ba/bahia/especial-publici...,BYD Conquista é líder nacional em market share...,<h2> Liderança nacional é impulsionada por alt...,G1,9807657.0,2026-02-10,BYD Conquista é líder nacional em market share...,byd,BYD Conquista é líder nacional em market share...,...,Promotores,Online,G1,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
3,5.980417e+18,https://www.uol.com.br/carros/noticias/redacao...,Preços passam de R$ 500 mil: veja os carros ch...,"Quando as marcas chinesas chegaram ao Brasil,...",Uol,4828868.0,2026-02-10,Preços passam de R$ 500 mil: veja os carros ch...,byd,"Em alguns casos, a sofisticação vai além e a...",...,Promotores,Online,Uol,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
4,1.251452e+19,https://www.uol.com.br/carros/noticias/redacao...,Volvo EX30 Ultra Twin: nova versão do SUV elét...,"Na estreia da temporada de 2026, o programa C...",Uol,4828868.0,2026-02-10,Volvo EX30 Ultra Twin: nova versão do SUV elét...,byd,"Medindo 4,61 metros de comprimento, 1,91 m d...",...,Promotores,Online,Uol,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1673,1.914170e+05,https://canaltech.com.br/carros/xiaomi-deixa-t...,40 seg Carros Xiaomi deixa Tesla para trás em ...,norte-americana a registrar o pior desempenho ...,Canaltech,274762.0,2025-11-13,40 seg Carros Xiaomi deixa Tesla para trás em ...,byd,"Vale destacar que, atualmente, a China tem um...",...,Inócuos,Online,Canaltech,None,Tier 1,Outros,Outros,Outros,Classificado,52213.02286
1674,1.699100e+05,https://www.poder360.com.br/poder-internaciona...,Toyota anuncia investimento de US$ 10 bilhões ...,A Toyota anunciou nesta 5ª feira (13.nov.2025...,Poder360,615574.0,2025-11-13,Toyota anuncia investimento de US$ 10 bilhões ...,byd,A iniciativa da Toyota busca responder a crít...,...,Inócuos,Online,Poder360,None,Tier 1,Outros,Outros,Outros,Classificado,116977.52722
1675,2.734357e+06,https://www.terra.com.br/mobilidade/novo-byd-y...,Novo BYD Yuan Plus é registrado no Brasil; con...,A BYD segue expandindo sua presença no mercad...,Terra Brasil,2595578.0,2025-11-13,Novo BYD Yuan Plus é registrado no Brasil; con...,byd,Novo BYD Yuan Plus é registrado no Brasil; con...,...,Promotores,Online,Terra Brasil,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000
1676,3.970518e+06,https://www.uol.com.br/carros/colunas/cacador-...,É fã da Honda? Confira 6 opções de usados da m...,Por mais que hoje a montadora japonesa seja a...,Uol,4828868.0,2025-11-13,É fã da Honda? Confira 6 opções de usados da m...,byd,Confira 6 opções de usados da marca com preço ...,...,Promotores,Online,Uol,None,Tier 1,Outros,Outros,Outros,Classificado,146818.00000


In [15]:
'df_classificado.parquet'.split('.parquet')[0]

'df_classificado'

In [5]:
dataframe.columns
dataframe.to_csv("teste.csv", index=False)
dataframe.to_parquet("teste.parquet", index=False)

In [8]:
gen_dataviews

<function src.structural_functions.goals_calculation.gen_dataviews(path_dataframe: str) -> pandas.core.frame.DataFrame>

In [6]:
gen_dataviews('teste.parquet')


,Data,Tipos de impactos,Sentimento,Empresa analisada,Produto analisado,Nível de Protagonismo final,Tier,Jornalista,Mídia,Ação,Tipo da ação,É ação,Status classificação,alcance,valoracao,count,jornalista_count,acao_count,ano,mes
0,2025-11-13,Inócuos,Neutro,byd,None,Citação relevante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,274762.0,52213.02286,1,1,0,2025,11
1,2025-11-13,Inócuos,Neutro,byd,None,Figurante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,274762.0,52213.02286,1,1,0,2025,11
2,2025-11-13,Inócuos,Positivo,byd,None,Figurante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,10486465.0,704249.52722,5,5,0,2025,11
3,2025-11-13,Promotores,Positivo,byd,None,Citação relevante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,13492666.0,753940.47192,6,6,0,2025,11
4,2025-11-13,Promotores,Positivo,byd,None,Protagonismo,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,4143416.0,417419.45167,4,4,0,2025,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,2026-02-10,Inócuos,Neutro,byd,None,Citação relevante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,4828868.0,146818.00000,1,1,0,2026,2
446,2026-02-10,Inócuos,Neutro,byd,None,Protagonismo,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,615574.0,116977.52722,1,1,0,2026,2
447,2026-02-10,Inócuos,Positivo,byd,None,Figurante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,352550.0,66995.07650,1,1,0,2026,2
448,2026-02-10,Promotores,Positivo,byd,None,Citação relevante,Tier 1,Outros,Online,Outros,Outros,sem ação,Classificado,4828868.0,146818.00000,1,1,0,2026,2


In [18]:
gen_goals_calculations('df_classificado_transformed.parquet')

{'NPS': {'NPS - Média mensal': {'descricao': 'O indicador é a média mensal do NPS. É ideal para observar o desempenho mensal, evitando desvios importantes no NPS e riscos de percepção da imagem da marca no período.',
   'baseline': 60.0,
   'conservadora': {'meta': 58.55, 'variacao': -1.45},
   'moderada': {'meta': 61.2, 'variacao': 1.2},
   'ousada': {'meta': 63.32, 'variacao': 3.32}},
  'NPS - Acumulado no ano': {'descricao': 'O indicador é o agregado do NPS no decorrer do ano. É ideal para observar o total da percepção no acumulada do ano sobre a marca.',
   'baseline': 57.5,
   'conservadora': {'meta': 56.00000000000001, 'variacao': -1.5},
   'moderada': {'meta': 59.0, 'variacao': 1.5},
   'ousada': {'meta': 60.0, 'variacao': 2.5}}},
 'Impressão Promotora': {'Impressão Promotora - Média mensal': {'descricao': 'O indicador é a média mensal do impacto promotor da marca. É ideal para observar o desempenho mensal da percepção promotora, com o objetivo de reduzir desvios importantes e m

In [11]:
x.keys()

dict_keys(['NPS', 'Impressão Promotora', 'Impressão Detratora', 'Protagonismo', 'Frequência', 'Valoração', 'Contr. ao NPS'])

In [12]:
x['Contr. ao NPS']

{'ações de comunicação': {'(%) Contr. das Ações ao NPS - Média mensal': {'descricao': 'É o indicador que mensura o quanto as ações de comunicação influenciam a construção da reputação da marca ao longo do período. A definição de uma média mensal como meta garante a manutenção das ações em um patamar adequado, evitando variações significativas ao longo dos meses.',
   'baseline': 99.82,
   'conservadora': {'meta': 99.76, 'variacao': -0.06},
   'moderada': {'meta': 101.82, 'variacao': 2.0},
   'ousada': {'meta': 99.88, 'variacao': 0.06}},
  '(%) Contr. das Ações ao NPS - Acumulado no ano': {'descricao': 'É o indicador que mensura o quanto as ações de comunicação influenciam a construção da reputação da marca ao longo do ano. A meta acumulada anual permite avaliar esse impacto de forma consolidada, reduzindo a influência de variações pontuais entre os meses.',
   'baseline': 99.65,
   'conservadora': {'meta': 97.66, 'variacao': -2.0},
   'moderada': {'meta': 101.64, 'variacao': 2.0},
   '